# 錯誤更正碼 1130

1116 與 1123 講義的 LDPC codes，  
靠隨機的 Tanner graph 與設計過的 degrees 接近 capacity。  
Arıkan（2009）的 **polar codes** 走另一條路：  
把 $N$ 個相同的 channels 合併再拆開，  
變成一部分幾乎完美、其餘幾乎全壞的 channels，  
再只用好的那些傳送資訊。

這是第一個 explicit 構造，  
被證明在所有 BMS channels 上都達到 capacity，  
encoding 與 decoding 都只要 $O(N\log N)$。  
它用的矩陣和 0914 講義的 RM codes 相同，  
5G 的 control channels 用的就是 polar codes。

食譜
- 兩個 bits 的變換：先解的 bit 經過 CN，後解的 bit 經過 VN。
- 遞迴 $m$ 次得到 $N = 2^m$ 個 bit channels，往兩極分化。
- Polar code：凍結不可靠的位置，$x = uG_N$。
- SC decoding：依序解每個 bit，用的是 1123 講義的兩條規則。
- 和 RM codes 用同一個矩陣，只是選 rows 的規則不同。

## Channel Polarization

### 兩個 bits 的變換

沿用 row-vector 慣例，令
$$F = \begin{bmatrix}1 & 0 \\ 1 & 1\end{bmatrix},\qquad
(x_1, x_2) = (u_1, u_2) F = (u_1 + u_2, u_2).$$
$x_1$ 與 $x_2$ 各自經過 BMS channel $W$（兩次獨立使用），  
收到 $y_1$ 與 $y_2$。  
令 $\mathsf a$ 是 $W$ 的 L-density。

Decoder 先猜 $u_1$，再猜 $u_2$。  
猜 $u_1$ 時 $u_2$ 未知，而 $u_1 = x_1 + x_2$：  
這是 1123 講義的串聯，$u_1$ 的 L-density 是 CN 的 $\mathsf a \boxdot \mathsf a$，  
記這個 channel 為 $W^-$。

猜 $u_2$ 時假設 $u_1$ 已經猜對。  
$y_2$ 看到 $u_2$，$y_1$ 看到 $u_1 + u_2$，相當於 $u_2$ 的另一份觀測：  
這是平行，$u_2$ 的 L-density 是 VN 的 $\mathsf a \odot \mathsf a$，  
記這個 channel 為 $W^+$。

在 $\operatorname{BEC}(\epsilon)$ 上，  
$u_1$ 要 $y_1$ 與 $y_2$ 都沒被擦除，$u_2$ 只要其中一個沒被擦除：
$$W^- = \operatorname{BEC}(2\epsilon - \epsilon^2),\qquad
W^+ = \operatorname{BEC}(\epsilon^2).$$
兩者的 capacity 加起來是 $(1 - \epsilon)^2 + (1 - \epsilon^2) = 2(1 - \epsilon)$，  
和原本兩次使用 $W$ 一樣：總 capacity 不變，只是重新分配。

一般的 BMS channel 也是如此：  
$C(W^-) + C(W^+) = 2C(W)$，  
而且 $C(W^-) \le C(W) \le C(W^+)$，這裡不證明。  
Bhattacharyya parameter 則可以由 1123 講義的結果直接得到：
$$\begin{aligned}
B(W^+) &= B(W)^2, \\
B(W) \le B(W^-) &\le 2B(W) - B(W)^2.
\end{aligned}$$

$B(W^+)$ 是 VN 的乘法公式，$B(W^-)$ 的上界是 CN 的上界。  
$B(W^-)$ 的下界來自 CN 的下界：
$$\sqrt{1 - (1 - B^2)^2} = B\sqrt{2 - B^2} \ge B.$$

### 遞迴

令 $N = 2^m$、$G_N = F^{\otimes m}$（Kronecker product），  
送出 $x = uG_N$，$u$ 的座標從 $0$ 編號到 $N - 1$。  
由 $G_N = F \otimes G_{N/2}$，把 $u$ 切成前後兩半 $u'$ 與 $u''$，  
令 $a = u'G_{N/2}$、$b = u''G_{N/2}$，則
$$x = (a + b, b).$$
這就是 0914 講義 RM codes 的 $(u, u + v)$，只是兩半的順序相反。

所以 $x$ 的第 $j$ 個與第 $N/2 + j$ 個座標，  
就是 $(a_j, b_j)$ 經過兩個 bits 的變換。  
先猜 $u'$：每個 $a_j$ 經過 $W^-$，  
$u'$ 面對的是用 $W^-$ 傳送、長度 $N/2$ 的同一個問題。  
猜對 $u'$ 之後再猜 $u''$：每個 $b_j$ 經過 $W^+$。

遞迴下去，$u_i$ 經過的 channel 由 $i$ 的二進位決定。  
把 $i$ 寫成 $m$ 位的二進位 $b_1b_2\cdots b_m$，$b_1$ 是最高位。  
從 $W$ 出發，依序對 $b_1, b_2, \ldots, b_m$ 套用變換，$0$ 是 $-$，$1$ 是 $+$。  
得到的 $N$ 個 channels 稱為 **bit channels**，記為 $W_N^{(i)}$。

在 $\operatorname{BEC}(\epsilon)$ 上，每個 bit channel 都是 BEC，  
令 $z_i$ 是 $W_N^{(i)}$ 的 erasure 機率。  
`bec_z` 每一層把每個 $z$ 換成相鄰的兩個數 $2z - z^2$ 與 $z^2$，  
相當於在索引的二進位最後多加一位。

和 1123 講義一樣，  
讓 SageMath 的數字常數維持 Python 的 `int` 與 `float`。

In [ ]:
import itertools
import numpy as np

Integer, RealNumber = int, float
rng = np.random.default_rng(1130)

In [ ]:
def bec_z(m, eps):
    z = np.array([eps])
    for _ in range(m):
        z = np.stack([2 * z - z**2, z**2], axis=1).ravel()
    return z

In [ ]:
print(bec_z(2, 0.5))

例如 $N = 4$、$\epsilon = 1/2$。  
$x = uG_4$ 是 $x_0 = u_0 + u_1 + u_2 + u_3$、$x_1 = u_1 + u_3$、  
$x_2 = u_2 + u_3$、$x_3 = u_3$。  
先猜的 $u_0 = x_0 + x_1 + x_2 + x_3$，  
四個 bits 都沒被擦除才知道：$z_0 = 1 - 1/16$。  
最後猜的 $u_3$，每個 $x_j$ 都看得到它：$z_3 = 1/16$。

### 極化

下圖是 $N = 1{,}024$、$\operatorname{BEC}(0.5)$ 的 $z_i$，  
左圖依索引排列，右圖由小到大排列。  
大部分的 $z_i$ 都接近 $0$ 或 $1$，  
這個現象稱為 **channel polarization**。

In [ ]:
z = bec_z(10, 0.5)
graphics_array([list_plot(list(z), size=2), list_plot(sorted(z), size=2)]).show(figsize=[10, 4])

$N$ 越大，分得越乾淨。  
下面對 $\operatorname{BEC}(0.5)$ 列出  
$m$、$z_i < 10^{-3}$ 的比例、$z_i < 10^{-9}$ 的比例，  
兩個比例都往 capacity $0.5$ 逼近。

In [ ]:
for m in [4, 8, 12, 16, 20]:
    z = bec_z(m, 0.5)
    print(m, np.round((z < 1e-3).mean(), 4), np.round((z < 1e-9).mean(), 4))

### 為什麼會極化

BEC 上有一個漂亮的論證。  
隨機選一個索引，也就是 $b_1, \ldots, b_m$ 各自是均勻的 $0$ 或 $1$。  
令 $Z_k$ 是套用前 $k$ 個變換後的 erasure 機率，$Z_0 = \epsilon$。  
$Z_{k+1}$ 各以機率 $1/2$ 是 $2Z_k - Z_k^2$ 或 $Z_k^2$，所以
$$\mathbb E[Z_{k+1} \mid Z_k] = \frac{(2Z_k - Z_k^2) + Z_k^2}2 = Z_k.$$

下一步的條件期望值等於現在的隨機過程，稱為 **martingale**。  
有界的 martingale 一定收斂，  
這是 martingale convergence theorem，這裡不證明。  
但每一步都移動 $|Z_{k+1} - Z_k| = Z_k(1 - Z_k)$，  
收斂就迫使 $Z_k(1 - Z_k) \to 0$：極限只能是 $0$ 或 $1$。

期望值一直是 $\epsilon$，  
所以極限是 $1$ 的機率是 $\epsilon$，是 $0$ 的機率是 $1 - \epsilon$。  
也就是說，$N \to \infty$ 時，  
幾乎完美的 bit channels 所佔的比例趨近 $1 - \epsilon$，正是 capacity。

一般的 BMS channel 上，capacity 同樣是 martingale，  
再配合 $B(W^+) = B(W)^2$，可以證明一樣會極化。  
Arıkan–Telatar（2009）進一步證明：對任意 $\beta < 1/2$，  
$B(W_N^{(i)}) < 2^{-N^\beta}$ 的索引所佔的比例趨近 capacity。

## Polar Codes

### Frozen bits 與 encoding

選 $K$ 個最可靠的索引當作 **information set** $\mathcal A$。  
其餘的 $u_i$ 固定為 $0$，稱為 **frozen bits**，decoder 事先知道它們。  
資訊放在 $u_{\mathcal A}$，送出 $x = uG_N$。  
這個 code 的 generator matrix 就是 $G_N$ 中 $\mathcal A$ 的 rows，  
rate 是 $K/N$。

Encoding 照遞迴 $x = (a + b, b)$ 計算，  
每一層做 $N/2$ 次加法，共 $m$ 層，所以是 $O(N\log N)$。  
下面檢查 `polar_encode(u)` 等於 $uG_N$。

In [ ]:
def polar_encode(u):
    if len(u) == 1:
        return u
    a, b = polar_encode(u[:len(u) // 2]), polar_encode(u[len(u) // 2:])
    return np.r_[(a + b) % 2, b]

In [ ]:
F = np.array([[1, 0], [1, 1]])

def kron_power(m):
    G = np.array([[1]])
    for _ in range(m):
        G = np.kron(G, F)
    return G

In [ ]:
u = rng.integers(2, size=16)
print(np.array_equal(polar_encode(u), u @ kron_power(4) % 2))

### Successive cancellation decoding

Decoder 依照同一個遞迴，用 LLR 計算。  
長度 $N$ 的問題，先用 CN 的 tanh rule 合併 $L_j$ 與 $L_{N/2+j}$，  
解長度 $N/2$ 的 $u'$。  
得到 $a = u'G_{N/2}$ 之後，用 VN 的規則
$$L''_j = L_{N/2+j} + (-1)^{a_j}L_j$$
解 $u''$。

到了長度 $1$：frozen bit 直接設為 $0$，否則依 LLR 的正負號判斷。  
這就是 **successive cancellation（SC）decoding**。  
每一層的計算量是 $O(N)$，共 $m$ 層，  
所以 decoding 也是 $O(N\log N)$。

`f` 是 tanh rule。  
`sc_decode` 傳回 $\hat u$ 與 $\hat u G_N$，後者是上一層計算 $a$ 時需要的。  
`leaves` 不是 `None` 時，依序記下每個 $u_i$ 判斷前的 LLR。

In [ ]:
def f(a, b):
    return 2 * np.arctanh(np.clip(np.tanh(a / 2) * np.tanh(b / 2), -1 + 1e-15, 1 - 1e-15))

In [ ]:
def sc_decode(L, frozen, leaves=None):
    if len(L) == 1:
        if leaves is not None:
            leaves.append(L[0])
        u = np.array([0 if frozen[0] else int(L[0] < 0)])
        return u, u
    h = len(L) // 2
    u1, a = sc_decode(f(L[:h], L[h:]), frozen[:h], leaves)
    u2, b = sc_decode(L[h:] + (1 - 2 * a) * L[:h], frozen[h:], leaves)
    return np.r_[u1, u2], np.r_[(a + b) % 2, b]

SC 的缺點是不能回頭：  
前面的 bit 猜錯了，後面的 bits 都用錯的 $a$ 計算。

### 用 genie 檢查 bit channels

若猜 $u_i$ 時，前面的 $u_0, \ldots, u_{i-1}$ 都已經正確，  
decoder 看到的正是 bit channel $W_N^{(i)}$。  
把所有 bits 都設為 frozen、傳送 all-zero codeword，就是這種情形。

下面在 $\operatorname{BEC}(0.5)$ 上取 $N = 64$，  
沒被擦除的 bit 的 LLR 設為 $30$，大到 $\tanh(15)$ 幾乎是 $1$。  
數出每個 $u_i$ 判斷前 LLR 為 $0$（被擦除）的比例，畫成紅點，  
和 `bec_z` 算出的 $z_i$（藍點）比較。

In [ ]:
N, trials = 64, 2000
erased = np.zeros(N)
for _ in range(trials):
    leaves = []
    sc_decode(np.where(rng.random(N) < 0.5, 0.0, 30.0), np.ones(N, bool), leaves)
    erased += np.array(leaves) == 0

In [ ]:
(list_plot(list(bec_z(6, 0.5)), legend_label=r"$z_i$")
 + list_plot(list(erased / trials), color="red", legend_label="simulation")
 ).show(figsize=6)

### 模擬：BEC

取 $N = 1{,}024$、$K = 512$，  
information set 是 $z_i$ 最小的 $512$ 個索引。  
SC 第一次猜錯的位置若是 $i$，那時前面都正確，  
所以 genie 版本的 $u_i$ 也會出錯。  
被擦除時用猜的，錯的機率是 $1/2$，  
因此 block error 的機率不超過 $\sum_{i\in\mathcal A} z_i/2$。

`sc_decode` 遇到 LLR 為 $0$ 時判為 $0$，  
所以要傳送隨機的資訊 bits，不能只送 all-zero codeword。  
`block_errors` 傳回 `trials` 個 blocks 中解錯的個數，  
`bec` 與 `bsc` 傳回模擬 channel 的函數。

In [ ]:
def info_frozen(z, K):
    frozen = np.ones(len(z), bool)
    frozen[np.argsort(z)[:K]] = False
    return frozen

In [ ]:
def block_errors(frozen, channel, trials=200):
    errors = 0
    for _ in range(trials):
        u = np.where(frozen, 0, rng.integers(2, size=len(frozen)))
        errors += np.any(sc_decode(channel(polar_encode(u)), frozen)[0] != u)
    return errors

In [ ]:
def bec(eps):
    return lambda x: np.where(rng.random(len(x)) < eps, 0.0, 30.0 * (1 - 2 * x))

def bsc(p):
    return lambda x: (1 - 2 * ((x + (rng.random(len(x)) < p)) % 2)) * np.log((1 - p) / p)

每一行依序是 $\epsilon$、union bound、模擬的 block error rate。

In [ ]:
for eps in [0.30, 0.35, 0.40, 0.45]:
    z = bec_z(10, eps)
    frozen = info_frozen(z, 512)
    print(eps, np.round(z[~frozen].sum() / 2, 4), block_errors(frozen, bec(eps)) / 200)

Block error rate 小的時候，union bound 很準。  
$\epsilon = 0.40$ 時約有四分之一的 blocks 失敗，  
和 1116 講義 $n = 500$ 的 $(3, 6)$ LDPC code 差不多。  
Polar codes 的長處是理論保證，  
有限長度的表現要靠延伸話題的 list decoding 補強。

### BMS channels 的構造

一般的 BMS channel 上，bit channels 不再是 BEC，  
精確計算每個 $B(W_N^{(i)})$ 很難。  
但 $B(W^+) = B^2$、$B(W^-) \le 2B - B^2$，而且兩個函數都遞增，  
所以對 $W$ 的 $B$ 跑 `bec_z`，得到每個 $B(W_N^{(i)})$ 的上界。  
這是 Arıkan 原始論文的構造方法。

1123 講義的 $P_e \le B/2$ 給出同樣的 union bound：  
block error 的機率不超過 $\sum_{i\in\mathcal A} B(W_N^{(i)})/2$。  
下面在 $\operatorname{BSC}(p)$ 上取 $N = 1{,}024$、$K = 512$，  
每一行依序是 $p$、union bound、模擬的 block error rate。

In [ ]:
for p in [0.02, 0.03, 0.04, 0.05, 0.06]:
    z = bec_z(10, 2 * np.sqrt(p * (1 - p)))
    frozen = info_frozen(z, 512)
    print(p, np.round(z[~frozen].sum() / 2, 4), block_errors(frozen, bsc(p)) / 200)

$p \ge 0.05$ 時 union bound 已經大於 $1$，但 SC 仍然常常成功。  
1116 講義長度 $1{,}000$ 的 $(3, 6)$ LDPC code 用 BP，  
$p = 0.06$ 時才偶爾失敗，這裡則有一成以上的 blocks 失敗。  
Tal–Vardy（2013）的方法能精確地估計每個 bit channel，  
選出更好的 information set。

## 和 RM Codes 比較

### 同一個矩陣的 rows

$F$ 的兩個 rows $(1, 0)$ 與 $(1, 1)$，是 $1 + x$ 與 $1$ 在 $x = 0, 1$ 的取值。  
Kronecker product 對應函數的乘積。

把 $G_N$ 的 column $c_1\cdots c_m$（二進位）  
對應到點 $(x_1, \ldots, x_m) = (c_1, \ldots, c_m)$，  
row $i = b_1\cdots b_m$ 就是
$$\prod_{j:b_j=0}(1 + x_j)$$
的取值向量。

它的 weight 是 $2^{\operatorname{wt}(i)}$，其中 $\operatorname{wt}(i)$
是 $i$ 的二進位中 $1$ 的個數：  
$b_j = 0$ 的每個 $x_j$ 都必須是 $0$，其餘的 $x_j$ 可以任意選。

$\operatorname{wt}(i) \ge m - r$ 的 rows 是最多 $r$ 個 $(1 + x_j)$ 的乘積，  
展開後次數都不超過 $r$。  
反過來，次數不超過 $r$ 的 monomial 也能寫成這些乘積的和，  
例如 $x_1x_2 = (1 + x_1)(1 + x_2) + (1 + x_1) + (1 + x_2) + 1$。  
所以這些 rows 生成的正是 0914 講義的 $\mathrm{RM}(r, m)$。

### 選 rows 的兩種規則

RM codes 依 weight 選 rows，  
polar codes 依 bit channel 的可靠度選 rows。  
下面取 $N = 32$、$K = 16$，  
比較 $\mathrm{RM}(2, 5)$ 與 $\operatorname{BEC}(0.5)$ 的 polar code 選的索引，  
最後一行是 $z_7$ 與 $z_{24}$。

In [ ]:
wt = np.array([bin(i).count("1") for i in range(32)])
rm_frozen = wt < 3
polar_frozen = info_frozen(bec_z(5, 0.5), 16)
print(np.where(~rm_frozen)[0])
print(np.where(~polar_frozen)[0])
print(np.round(bec_z(5, 0.5)[[7, 24]], 3))

兩者只差一個 row。  
RM 選了 row $7 = 00111$，weight 是 $8$，但 $z_7 \approx 0.6$；  
polar 改選 row $24 = 11000$，weight 只有 $4$，  
但 $z_{24} \approx 0.4$ 比較小。

$G_N$ 中任意一組 rows 生成的 code，  
minimum distance 就是這些 rows 中最小的 weight，這裡不證明。  
所以 $\mathrm{RM}(2, 5)$ 的 $d = 8$，polar code 的 $d$ 只有 $4$。  
下面窮舉兩個 codes 的 $2^{16}$ 個 codewords 驗證。

In [ ]:
msgs = np.array(list(itertools.product([0, 1], repeat=16)))
for frozen in [rm_frozen, polar_frozen]:
    print((msgs[1:] @ kron_power(5)[~frozen] % 2).sum(axis=1).min())

### SC 與 ML

1116 講義說明了：在 BEC 上，  
ML decoding 失敗當且僅當沒被擦除的 columns 的 rank 小於 $K$。  
`gf2_rank` 用高斯消去法計算 $\mathbb F_2$ 上的 rank，  
`ml_errors` 傳回 `trials` 個 blocks 中 ML 失敗的個數。

In [ ]:
def gf2_rank(A):
    A, r = A % 2, 0
    for c in range(A.shape[1]):
        piv = np.nonzero(A[r:, c])[0]
        if len(piv) == 0:
            continue
        A[[r, r + piv[0]]] = A[[r + piv[0], r]]
        rows = (A[:, c] == 1) & (np.arange(len(A)) != r)
        A[rows] = (A[rows] + A[r]) % 2
        r += 1
        if r == len(A):
            break
    return r

In [ ]:
def ml_errors(frozen, eps, trials=2000):
    Gk = kron_power(5)[~frozen]
    return sum(gf2_rank(Gk[:, rng.random(32) >= eps]) < len(Gk) for _ in range(trials))

每一行依序是 $\epsilon$ 與 code，  
接著是 SC 的 block error rate，最後是 ML 的。

In [ ]:
for eps in [0.2, 0.3, 0.4]:
    for name, frozen in [("RM", rm_frozen), ("polar", polar_frozen)]:
        print(eps, name, block_errors(frozen, bec(eps), 2000) / 2000, ml_errors(frozen, eps) / 2000)

SC 下 polar code 比較好，因為它正是為 SC 挑選的；  
ML 下 RM code 比較好，因為它的 minimum distance 比較大。

Kudekar 等人（2017）證明：  
RM codes 在 BEC 上用 ML decoding 達到 capacity，  
Abbe–Sandon（2023）推廣到所有 BMS channels。  
但 RM codes 沒有像 SC 這樣快、又能達到 capacity 的 decoder。

## 應用

### 5G NR 的 control channels

1116 講義提到 5G 的資料通道用 LDPC codes。  
控制訊息（例如排程資訊）很短，5G NR 改用 polar codes：  
上行最長 $N = 1{,}024$，下行最長 $N = 512$。  
所有長度共用一條長度 $1{,}024$ 的可靠度排序，  
依序取出最可靠的位置當作 information set，  
不必針對 channel 重新計算。

資訊 bits 後面接上 CRC，  
decoder 用下面的 SC list decoding，  
從 list 中挑出 CRC 正確的候選。

## 延伸話題

### SC list decoding

SC 每遇到一個資訊 bit，只留下一個猜測。  
Tal–Vardy（2015）提出 **SC list decoding**：  
同時保留好幾條路徑，例如 $8$ 條。  
每遇到一個資訊 bit，每條路徑分成兩條，  
再留下 likelihood 最大的那幾條。  
最後選出通過 CRC 的路徑，  
有限長度的表現就能和 LDPC codes 競爭。

### Kernels 與 PAC codes

把 $F$ 換成更大的 kernel，可以極化得更快。  
Arıkan（2019）的 PAC（polarization-adjusted convolutional）codes，  
在 polar 變換之前先做一次 convolutional 編碼，短長度時表現更好。

## 週五作業

### 週一作業